# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The lane ranks visible pages by how likely they are to be declining, so I want a good ranker, not a classifier tuned for raw accuracy. I take **Logistic Regression** as the model to beat the baseline with, and a **Random Forest** as a complexity check: if the extra capacity earns nothing, that is itself the finding.

The signals are mostly monotonic and the label is noisy, so a linear model with class weighting is a natural first choice. Clustering and plain correlation analysis were also on the menu, but the decision here is a ranked queue judged against a labeled outcome, which is a supervised ranking problem. Features are the leakage-safe set from my data contract: prior-window traffic and static page properties only, never the trend or 30-day outcome columns.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
v = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)

num = ["content_age_days", "days_since_last_update", "word_count", "char_count",
       "search_volume", "competition", "cpc",
       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
v["has_keyword"] = v["search_volume"].notna().astype(int)
v["has_word_count"] = v["word_count"].notna().astype(int)
for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    v["log_" + c] = np.log1p(v[c].fillna(0))
cat = pd.get_dummies(v[["content_type", "main_intent", "competition_level"]].fillna("unknown"), drop_first=True)
extra = ["has_keyword", "has_word_count", "log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d"]
X = pd.concat([v[num].fillna(0), v[extra], cat], axis=1)
y = v["is_declining"].values
groups = v["client_id"].values

print("pages:", len(v), " features:", X.shape[1], " base rate declining:", round(y.mean(), 3))

pages: 22006  features: 24  base rate declining: 0.598


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Client-grouped. A page carries its client's fingerprint: shared templates, one niche, the same tracking setup. A random split lets a model memorize the client and report skill it does not have, so the honest question is whether it works on a client it has never seen. I split by `client_id` with `GroupShuffleSplit` and hold out 30% of clients. Time-aware splitting would be better, but the starter slice is a single snapshot, so grouped is the honest split available here. Every number below comes from this one held-out split.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(X, y, groups))
overlap = set(groups[tr]) & set(groups[te])

print("train pages:", len(tr), " test pages:", len(te))
print("train clients:", len(set(groups[tr])), " test clients:", len(set(groups[te])))
print("clients in both train and test:", len(overlap))
print("test base rate:", round(y[te].mean(), 3))

train pages: 17251  test pages: 4755
train clients: 21  test clients: 9
clients in both train and test: 0
test base rate: 0.577


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same visible population, same client-grouped test split, same metrics as Week 4: ROC-AUC and Precision@50, with the base rate beside them. The baseline is my Week-4 CTR-fix rule score, evaluated here as a ranker for decline. I add a naive staleness ranker as a floor. The model has to beat these, not just look more sophisticated.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return np.asarray(labels)[order[:k]].mean()

gp = v[(v["avg_position"] > 0) & (v["position_tier"].isin(["top_3", "page_1", "striking"]))]
exp_ctr = gp.groupby("position_tier")["ctr"].median()
v["gap"] = v["position_tier"].map(exp_ctr) - v["ctr"]
ctrfix = np.where((v["avg_position"] > 0) & (v["position_tier"].isin(["top_3", "page_1", "striking"])) & (v["gap"] > 0),
                  v["impressions_90d"] * v["gap"], 0.0)

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
logreg.fit(X.iloc[tr], y[tr])
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=1, class_weight="balanced")
rf.fit(X.iloc[tr], y[tr])

yte = y[te]
rows = []
for name, s in [("baseline: Week-4 CTR-fix", ctrfix[te]),
                ("baseline: staleness", v["days_since_last_update"].values[te]),
                ("model: Random Forest", rf.predict_proba(X.iloc[te])[:, 1]),
                ("model: Logistic Regression", logreg.predict_proba(X.iloc[te])[:, 1])]:
    rows.append({"method": name, "ROC_AUC": round(roc_auc_score(yte, s), 3), "P@50": round(precision_at_k(s, yte), 3)})
table = pd.DataFrame(rows)
print("test base rate:", round(yte.mean(), 3))
print(table.to_string(index=False))

test base rate: 0.577
                    method  ROC_AUC  P@50
  baseline: Week-4 CTR-fix    0.569  0.54
       baseline: staleness    0.431  0.62
      model: Random Forest    0.580  0.58
model: Logistic Regression    0.601  0.84


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Logistic Regression is the model I keep: it beats the Week-4 baseline on both metrics, and the Random Forest's extra capacity does not improve on it, so complexity is not paying off here. Permutation importance on the held-out split shows what it leans on. Then I read the errors at the top of the queue, where a wrong call costs an editor real time.

In [4]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(logreg, X.iloc[te], yte, scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=1)
top = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(8)
print("permutation importance (drop in ROC-AUC when shuffled), top 8:")
print(top.round(4).to_string())

probs = logreg.predict_proba(X.iloc[te])[:, 1]
order = np.argsort(-probs)[:50]
top_idx = te[order]
top_y = y[top_idx]
fp = top_idx[top_y == 0]
tp = top_idx[top_y == 1]
print(f"\ntop-50 queue: {len(tp)} true declines, {len(fp)} false positives")
print("false positives vs true positives (median prior-window signals):")
print(f"  impressions_prev_30d   fp {v.loc[fp,'impressions_prev_30d'].median():.0f}   tp {v.loc[tp,'impressions_prev_30d'].median():.0f}")
print(f"  days_since_last_update  fp {v.loc[fp,'days_since_last_update'].median():.0f}   tp {v.loc[tp,'days_since_last_update'].median():.0f}")

permutation importance (drop in ROC-AUC when shuffled), top 8:
log_clicks_prev_30d             0.0862
content_age_days                0.0382
log_sessions_prev_30d           0.0277
competition_level_unknown       0.0246
content_type_keyword article    0.0173
has_keyword                     0.0165
content_type_feedly article     0.0146
competition_level_LOW           0.0115

top-50 queue: 42 true declines, 8 false positives
false positives vs true positives (median prior-window signals):
  impressions_prev_30d   fp 1068   tp 2322
  days_since_last_update  fp 20   tp 20


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.